In [1]:
# imports

import pandas as pd
import numpy as np
import json

In [6]:
# Helpers to read in data and convert to dataframe

def load_json_data(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        return json.load(f)

def convert_to_dataframe(data, is_full_cleaned=False):
    record = []
    for table_id, details in data.items():
        statements = details[0]
        enatiled_statuses = details[1]
        caption = details[-1]
        
        if is_full_cleaned:
            pos_tags = details[2]
            for statement, entailed_status, statement_pos_tags in zip(statements, enatiled_statuses, pos_tags):
                record.append({
                    "table_id": table_id,
                    "statement": statement,
                    "entailed_status": entailed_status,
                    "caption": caption,
                    "pos_tags": statement_pos_tags
                }) 
        else:
            for statement, entailed_status in zip(statements, enatiled_statuses):
                record.append({
                    "table_id": table_id,
                    "statement": statement,
                    "entailed_status": entailed_status,
                    "caption": caption
                })
    return pd.DataFrame(record)

def load_and_convert_data_to_dataframe(path, is_full_cleaned=False):
    raw_data = load_json_data(path)
    
    return convert_to_dataframe(raw_data, is_full_cleaned=is_full_cleaned)

In [7]:
# Load in data frames

df_full_cleaned = load_and_convert_data_to_dataframe("data/full_cleaned.json", is_full_cleaned=True)

df_r1_training = load_and_convert_data_to_dataframe("data/r1_training_all.json")

df_r2_training = load_and_convert_data_to_dataframe("data/r2_training_all.json")


In [8]:
# Replicate their simple and complex channel data analysis: gathering

# Identify unique tables
tables_in_full_cleaned = df_full_cleaned["table_id"].unique()
tables_in_r1 = df_r1_training["table_id"].unique()
tables_in_r2 = df_r2_training["table_id"].unique()

# Find tables that are in r1/r2 and in full cleaned
tables_in_r1_and_full_cleaned = np.intersect1d(tables_in_r1, tables_in_full_cleaned)
tables_in_r2_and_full_cleaned = np.intersect1d(tables_in_r2, tables_in_full_cleaned)

# Identify r1 and r2 statements in full cleaned
statements_in_full_cleaned_from_r1_tables = df_full_cleaned[df_full_cleaned["table_id"].isin(tables_in_r1_and_full_cleaned)]
statements_in_full_cleaned_from_r2_tables = df_full_cleaned[df_full_cleaned["table_id"].isin(tables_in_r2_and_full_cleaned)]

# Average length of entailed statements
full_cleaned_entailed_statements = df_full_cleaned[df_full_cleaned["entailed_status"] == 1]["statement"]
average_length_full_cleaned_entailed_statements = full_cleaned_entailed_statements.str.split().apply(len).mean()

full_cleaned_entailed_statements_from_r1_tables = statements_in_full_cleaned_from_r1_tables[statements_in_full_cleaned_from_r1_tables["entailed_status"] == 1]["statement"]
average_length_full_cleaned_entailed_statements_from_r1_tables = full_cleaned_entailed_statements_from_r1_tables.str.split().apply(len).mean()

full_cleaned_entailed_statements_from_r2_tables = statements_in_full_cleaned_from_r2_tables[statements_in_full_cleaned_from_r2_tables["entailed_status"] == 1]["statement"]
average_length_full_cleaned_entailed_statements_from_r2_tables = full_cleaned_entailed_statements_from_r2_tables.str.split().apply(len).mean()


# Average length of refuted statements
full_cleaned_refuted_statements = df_full_cleaned[df_full_cleaned["entailed_status"] == 0]["statement"]
average_length_full_cleaned_refuted_statements = full_cleaned_refuted_statements.str.split().apply(len).mean()

full_cleaned_refuted_statements_from_r1_tables = statements_in_full_cleaned_from_r1_tables[statements_in_full_cleaned_from_r1_tables["entailed_status"] == 0]["statement"]
average_length_full_cleaned_refuted_statements_from_r1_tables = full_cleaned_refuted_statements_from_r1_tables.str.split().apply(len).mean()

full_cleaned_refuted_statements_from_r2_tables = statements_in_full_cleaned_from_r2_tables[statements_in_full_cleaned_from_r2_tables["entailed_status"] == 0]["statement"]
average_length_full_cleaned_refuted_statements_from_r2_tables = full_cleaned_refuted_statements_from_r2_tables.str.split().apply(len).mean()

In [9]:
# Replicate their simple and complex channel data analysis: displaying results

# Tables
print(f"Number of unique tables in full cleaned dataset: {len(tables_in_full_cleaned)}")
print(f"Number of unique tables in r1 training dataset: {len(tables_in_r1)}")
print(f"Number of unique tables in r2 training dataset: {len(tables_in_r2)}")

# Number of statements
print(f"Number of statements in full cleaned dataset: {len(df_full_cleaned)}")
print(f"Number of statements in r1 training dataset: {len(statements_in_full_cleaned_from_r1_tables)}")
print(f"Number of statements in r2 training dataset: {len(statements_in_full_cleaned_from_r2_tables)}")

# Statement length of entailed statements
print(f"Average length of entailed statements in full cleaned dataset: {average_length_full_cleaned_entailed_statements:.2f} words")
print(f"Average length of entailed statements in r1 training dataset: {average_length_full_cleaned_entailed_statements_from_r1_tables:.2f} words")
print(f"Average length of entailed statements in r2 training dataset: {average_length_full_cleaned_entailed_statements_from_r2_tables:.2f} words")

# Statement length of refuted statements
print(f"Average length of refuted statements in full cleaned dataset: {average_length_full_cleaned_refuted_statements:.2f} words")
print(f"Average length of refuted statements in r1 training dataset: {average_length_full_cleaned_refuted_statements_from_r1_tables:.2f} words")
print(f"Average length of refuted statements in r2 training dataset: {average_length_full_cleaned_refuted_statements_from_r2_tables:.2f} words")


Number of unique tables in full cleaned dataset: 16573
Number of unique tables in r1 training dataset: 9181
Number of unique tables in r2 training dataset: 7392
Number of statements in full cleaned dataset: 117854
Number of statements in r1 training dataset: 49850
Number of statements in r2 training dataset: 68004
Average length of entailed statements in full cleaned dataset: 13.71 words
Average length of entailed statements in r1 training dataset: 13.05 words
Average length of entailed statements in r2 training dataset: 14.19 words
Average length of refuted statements in full cleaned dataset: 13.79 words
Average length of refuted statements in r1 training dataset: 13.32 words
Average length of refuted statements in r2 training dataset: 14.15 words
